# Principal Component Analysis (PCA)

This notebook demonstrates Principal Component Analysis on the Breast Cancer dataset.
PCA is a dimensionality reduction technique that identifies the directions of maximum variance in the data.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load and Explore the Data

In [ ]:
# Load the dataset
df = pd.read_csv("breast-cancer.csv")
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
df.head()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum().sum())

# Display basic statistics
print("\nDataset Info:")
print(f"Total samples: {df.shape[0]}")
print(f"Total features: {df.shape[1]}")
print(f"\nDiagnosis distribution:")
print(df['diagnosis'].value_counts())

## 2. Data Preprocessing

In [ ]:
# Encode categorical variable (diagnosis)
df_encoded = pd.get_dummies(df, dtype=int)
print("Encoded dataset shape:", df_encoded.shape)
print("\nFirst few rows after encoding:")
df_encoded.head()

In [ ]:
# Check for missing values after encoding
missing_values = df_encoded.isnull().sum()
if missing_values.sum() == 0:
    print("✓ No missing values found")
else:
    print("Missing values:")
    print(missing_values[missing_values > 0])

In [ ]:
# Remove outliers using IQR method
def remove_outliers(data):
    """
    Remove outliers using Interquartile Range (IQR) method
    """
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - (1.5 * IQR)
    upper_limit = Q3 + (1.5 * IQR)
    return data[(data >= lower_limit) & (data <= upper_limit)]

# Apply outlier removal
print(f"Original dataset shape: {df_encoded.shape}")
for col in df_encoded.columns:
    df_encoded = df_encoded[(df_encoded[col] >= df_encoded[col].quantile(0.25) - 1.5 * (df_encoded[col].quantile(0.75) - df_encoded[col].quantile(0.25))) & 
                             (df_encoded[col] <= df_encoded[col].quantile(0.75) + 1.5 * (df_encoded[col].quantile(0.75) - df_encoded[col].quantile(0.25)))]

print(f"Dataset shape after outlier removal: {df_encoded.shape}")
print(f"Rows removed: {569 - df_encoded.shape[0]}")

## 3. Feature Standardization

In [ ]:
# Extract features and standardize
X = df_encoded.values
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

print("Standardized data statistics:")
print(f"Mean: {X_std.mean():.6f}")
print(f"Std Dev: {X_std.std():.6f}")
print(f"Data shape: {X_std.shape}")

## 4. Apply PCA

In [ ]:
# Apply PCA to all components
pca = PCA()
principal_components = pca.fit_transform(X_std)

# Create DataFrame with principal components
principal_df = pd.DataFrame(
    data=principal_components,
    columns=[f'PC{i+1}' for i in range(principal_components.shape[1])]
)

print(f"Principal components shape: {principal_df.shape}")
print("\nFirst few rows of principal components:")
principal_df.head()

## 5. Analyze Explained Variance

In [ ]:
# Calculate explained variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# Display variance information
print("Explained Variance by Component:")
print("="*50)
for i, (ind, cum) in enumerate(zip(explained_variance[:10], cumulative_variance[:10])):
    print(f"PC{i+1}: {ind:.4f} (Cumulative: {cum:.4f})")

# Find number of components for 95% variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f"\nComponents needed for 95% variance: {n_components_95}")
print(f"Total variance explained: {cumulative_variance[-1]:.4f}")

## 6. Visualize Scree Plot

In [ ]:
# Create scree plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot individual explained variance
ax1.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.6, color='steelblue', label='Individual')
ax1.set_xlabel('Principal Component', fontsize=12, fontweight='bold')
ax1.set_ylabel('Explained Variance Ratio', fontsize=12, fontweight='bold')
ax1.set_title('Individual Explained Variance by PC', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Plot cumulative explained variance
ax2.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 'o-', color='darkred', linewidth=2, markersize=4, label='Cumulative')
ax2.axhline(y=0.95, color='green', linestyle='--', linewidth=2, label='95% Variance')
ax2.axvline(x=n_components_95, color='orange', linestyle='--', linewidth=2, label=f'{n_components_95} Components')
ax2.set_xlabel('Number of Principal Components', fontsize=12, fontweight='bold')
ax2.set_ylabel('Cumulative Explained Variance', fontsize=12, fontweight='bold')
ax2.set_title('Cumulative Explained Variance', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Visualize PC1 vs PC2

In [ ]:
# Create scatter plot for PC1 vs PC2
plt.figure(figsize=(10, 8))

# Add diagnosis information for coloring
diagnosis_values = df_encoded['diagnosis_M'].values
colors = ['red' if x == 1 else 'blue' for x in diagnosis_values]

scatter = plt.scatter(principal_df['PC1'], principal_df['PC2'], c=colors, alpha=0.6, s=50)

plt.xlabel(f'PC1 ({explained_variance[0]:.2%} variance)', fontsize=12, fontweight='bold')
plt.ylabel(f'PC2 ({explained_variance[1]:.2%} variance)', fontsize=12, fontweight='bold')
plt.title('PCA: PC1 vs PC2 (Colored by Diagnosis)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Create legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='red', alpha=0.6, label='Malignant'),
                   Patch(facecolor='blue', alpha=0.6, label='Benign')]
plt.legend(handles=legend_elements, loc='best')

plt.tight_layout()
plt.show()

## 8. Feature Contribution Analysis

In [ ]:
# Get feature contributions for first 3 PCs
loadings = pca.components_[:3].T * np.sqrt(pca.explained_variance_[:3])

# Create heatmap
fig, ax = plt.subplots(figsize=(10, 8))
feature_names = [f'F{i+1}' for i in range(loadings.shape[0])]
component_names = ['PC1', 'PC2', 'PC3']

sns.heatmap(loadings, 
            xticklabels=component_names,
            yticklabels=feature_names,
            cmap='coolwarm',
            center=0,
            cbar_kws={'label': 'Loading'},
            ax=ax)

ax.set_title('Feature Contributions to Principal Components', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Feature loadings shape:", loadings.shape)

## 9. 3D Visualization

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Create 3D plot
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot points
scatter = ax.scatter(principal_df['PC1'], principal_df['PC2'], principal_df['PC3'],
                      c=colors, alpha=0.6, s=50)

ax.set_xlabel(f'PC1 ({explained_variance[0]:.2%})', fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC2 ({explained_variance[1]:.2%})', fontsize=11, fontweight='bold')
ax.set_zlabel(f'PC3 ({explained_variance[2]:.2%})', fontsize=11, fontweight='bold')
ax.set_title('3D PCA Visualization', fontsize=14, fontweight='bold')

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='red', alpha=0.6, label='Malignant'),
                   Patch(facecolor='blue', alpha=0.6, label='Benign')]
ax.legend(handles=legend_elements, loc='best')

plt.tight_layout()
plt.show()

## 10. Summary Statistics

In [ ]:
# Summary of PCA analysis
print("\n" + "="*60)
print("PCA ANALYSIS SUMMARY")
print("="*60)
print(f"Original features: {X_std.shape[1]}")
print(f"Original samples: {X_std.shape[0]}")
print(f"\nComponents for 90% variance: {np.argmax(cumulative_variance >= 0.90) + 1}")
print(f"Components for 95% variance: {np.argmax(cumulative_variance >= 0.95) + 1}")
print(f"Components for 99% variance: {np.argmax(cumulative_variance >= 0.99) + 1}")
print(f"\nDimensionality reduction: {X_std.shape[1]} -> {n_components_95} features")
print(f"Reduction ratio: {((X_std.shape[1] - n_components_95) / X_std.shape[1] * 100):.2f}%")
print("="*60)